# WarriorMomentum Strategy Analysis

This notebook analyzes the WarriorMomentum strategy across all available USDT pairs.
The strategy is designed to capture momentum moves of 20-30% in cryptos with strong daily charts and high relative volume.

## Key Features:
- Bull Flag and Flat Top Breakout patterns
- High relative volume (2x+ above average)
- Strong daily charts (above moving averages)
- Partial exits at profit targets
- Dynamic stop loss management


## Setup

### Change Working directory to repository root


In [71]:
import os
from pathlib import Path

# Change directory
# Modify this cell to insure that the output shows the correct path.
# Define all paths relative to the project root shown in the cell output
project_root = "/Users/conradlz/Documents/webclones/freqtrade"
i = 0
try:
    os.chdir(project_root)
    if not Path("LICENSE").is_file():
        i = 0
        while i < 4 and (not Path("LICENSE").is_file()):
            os.chdir(Path(Path.cwd(), "../"))
            i += 1
        project_root = Path.cwd()
except FileNotFoundError:
    print("Please define the project root relative to the current directory")
print(Path.cwd())


/Users/conradlz/Documents/webclones/freqtrade


In [72]:
from freqtrade.configuration import Configuration
from freqtrade.enums import RunMode
from freqtrade.resolvers import ExchangeResolver

# Initialize configuration
config = Configuration.from_files(["user_data/config.json"])

# Define constants for WarriorMomentum strategy
config["timeframe"] = "5m"
config["strategy"] = "WarriorMomentum"
config["stake_currency"] = "USD"
config["dry_run"] = True
config["dataformat_ohlcv"] = "feather"
config["runmode"] = RunMode.BACKTEST

# Data location
data_location = config["datadir"]

print(f"Strategy: {config['strategy']}")
print(f"Timeframe: {config['timeframe']}")
print(f"Data directory: {data_location}")
print(f"Stake currency: {config['stake_currency']}")
print(f"Max open trades: {config['max_open_trades']}")
print(f"Stake amount: {config['stake_amount']}")


2025-07-21 14:55:09,553 - freqtrade.configuration.load_config - INFO - Using config: user_data/config.json ...

2025-07-21 14:55:09,562 - freqtrade.loggers - INFO - Enabling colorized output.

2025-07-21 14:55:09,563 - freqtrade.loggers - INFO - Logfile configured

2025-07-21 14:55:09,568 - freqtrade.loggers - INFO - Verbosity set to 0

2025-07-21 14:55:09,572 - freqtrade.configuration.configuration - INFO - Using user-data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data ...

2025-07-21 14:55:09,575 - freqtrade.configuration.configuration - INFO - Using data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken ...

2025-07-21 14:55:09,578 - freqtrade.exchange.check_exchange - INFO - Checking exchange...

2025-07-21 14:55:09,586 - freqtrade.exchange.check_exchange - INFO - Exchange "kraken" is officially supported by the Freqtrade development team.

Strategy: WarriorMomentum
Timeframe: 5m
Data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
Stake currency: USD
Max open trades: 5
Stake amount: unlimited


In [73]:
import glob
import re
from pathlib import Path

def find_pairs(data_dir, timeframe="5m"):
    """Find all USDT pairs available in the data directory."""
    data_path = Path(data_dir)
    usdt_pairs = []

    # Search for feather files with USDT pairs
    pattern = f"*USD-{timeframe}.feather"

    # Search in all subdirectories (different exchanges)
    print(data_path)
    if data_path.is_dir():
        files = list(data_path.glob(pattern))
        print(files)
        for file in files:
            # Extract pair name from filename
            pair_name = file.stem.replace(f"-{timeframe}", "").replace("_", "/")
            print(pair_name)
            if pair_name not in usdt_pairs:
                usdt_pairs.append(pair_name)
    else:
        print(f"Not a directory: {data_path}")

    return sorted(usdt_pairs)

# Find all USDT pairs
usd_pairs = find_pairs(data_location, config["timeframe"])

print(f"Found {len(usd_pairs)} USD pairs:")
for i, pair in enumerate(usd_pairs, 1):
    print(f"{i:2d}. {pair}")

if not usd_pairs:
    print("No USD pairs found. Please download data first using:")
    print("freqtrade download-data --exchange binance --pairs BTC/USDT ETH/USDT --timeframes 5m 1d")
    print("Or use the exchange's available pairs list instead.")


/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
[PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/ETC_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/USDD_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/IDEX_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/POWR_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/PENGU_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/BOBA_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/PUMP_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/JASMY_USD-5m.feather'), PosixPath('/Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken/SPK_USD-5m.feather'), PosixPath('/U

In [74]:
# If no local data found, get pairs from exchange
if not usd_pairs:
    try:
        # Initialize exchange
        exchange = ExchangeResolver.load_exchange(config, validate=False)

        # Get all available pairs
        all_pairs = exchange.get_markets().keys()

        # Filter for USDT pairs
        usdt_pairs = [pair for pair in all_pairs if pair.endswith('/USDT')]
        usdt_pairs = sorted(usdt_pairs)

        print(f"Found {len(usdt_pairs)} USDT pairs from exchange:")
        for i, pair in enumerate(usdt_pairs[:20], 1):  # Show first 20
            print(f"{i:2d}. {pair}")

        if len(usdt_pairs) > 20:
            print(f"... and {len(usdt_pairs) - 20} more pairs")

    except Exception as e:
        print(f"Error getting pairs from exchange: {e}")
        # Fallback to common USDT pairs
        usdt_pairs = [
            "BTC/USDT", "ETH/USDT", "BNB/USDT", "ADA/USDT", "XRP/USDT",
            "SOL/USDT", "DOT/USDT", "DOGE/USDT", "AVAX/USDT", "SHIB/USDT",
            "MATIC/USDT", "LTC/USDT", "UNI/USDT", "LINK/USDT", "ATOM/USDT",
            "XLM/USDT", "BCH/USDT", "NEAR/USDT", "ALGO/USDT", "VET/USDT"
        ]
        print(f"Using fallback list of {len(usdt_pairs)} common USDT pairs")

print(f"\nTotal USDT pairs to analyze: {len(usdt_pairs)}")



Total USDT pairs to analyze: 445


In [75]:
from freqtrade.enums.candletype import CandleType
from freqtrade.optimize.backtesting import Backtesting
from freqtrade.data.history import load_pair_history
import asyncio
import nest_asyncio

# Enable nested event loops for Jupyter notebooks
nest_asyncio.apply()

# Configure backtest settings for multi-pair analysis
config_backtest = config.copy()
config_backtest.update({
    "timerange": "",  # Use all available data
    "stake_currency": "USD",
    "dry_run": True,
    "dataformat_ohlcv": "feather",
    "runmode": RunMode.BACKTEST,

    "pairlists": [{"method": "StaticPairList", "pairlist": usd_pairs}],

    "exchange": {
        "name": "kraken",
        "pair_whitelist": usd_pairs,
    },
    "position_adjustment_enable": True,  # Enable for WarriorMomentum strategy
    "max_open_trades": 10,  # Increase for multi-pair testing,
    "candle_type_def": CandleType.SPOT,
})

print("=== BACKTEST CONFIGURATION ===")
print(f"Strategy: {config_backtest['strategy']}")
print(f"Timeframe: {config_backtest['timeframe']}")
print(f"Pairs: {len(usd_pairs)} USD pairs")
print(f"Max open trades: {config_backtest['max_open_trades']}")
print(f"Stake amount: {config_backtest['stake_amount']} {config_backtest['stake_currency']}")
print(f"Position adjustment: {config_backtest['position_adjustment_enable']}")
print(f"Data directory: {config_backtest['datadir']}")
print(f"Data format: {config_backtest['dataformat_ohlcv']}")

# Initialize exchange if not already done
if 'exchange' not in locals():
    exchange = ExchangeResolver.load_exchange(config_backtest, validate=False)

print("\n=== INITIALIZING BACKTEST ===")
print("Setting up backtesting engine...")

# Initialize backtesting
backtesting = Backtesting(config_backtest, exchange)

print("Backtest engine initialized successfully!")


=== BACKTEST CONFIGURATION ===
Strategy: WarriorMomentum
Timeframe: 5m
Pairs: 445 USD pairs
Max open trades: 10
Stake amount: unlimited USD
Position adjustment: True
Data directory: /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken
Data format: feather

=== INITIALIZING BACKTEST ===
Setting up backtesting engine...


2025-07-21 14:55:09,660 - freqtrade.resolvers.iresolver - INFO - Using resolved strategy WarriorMomentum from '/Users/conradlz/Documents/webclones/freqtrade/user_data/strategies/WarriorMomentum.py'...

2025-07-21 14:55:09,662 - freqtrade.strategy.hyper - INFO - Found no parameter file.

2025-07-21 14:55:09,663 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'timeframe' with value in config file: 5m.

2025-07-21 14:55:09,664 - freqtrade.resolvers.strategy_resolver - INFO - Override strategy 'stake_currency' with value in config file: USD.

...

2025-07-21 14:55:09,713 - freqtrade.resolvers.strategy_resolver - INFO - Strategy using max_open_trades: 10

2025-07-21 14:55:09,717 - freqtrade.configuration.config_validation - INFO - Validating configuration ...

2025-07-21 14:55:09,768 - freqtrade.resolvers.iresolver - INFO - Using resolved pairlist StaticPairList from 
'/Users/conradlz/Documents/webclones/freqtrade/freqtrade/plugins/pairlist/StaticPairList.py'...

2025-07-21 14:55:10,448 - freqtrade.optimize.backtesting - INFO - Using fee 0.4000% - worst case fee from exchange (lowest tier).

Backtest engine initialized successfully!


In [76]:
print("Running backtest...")
print(f"Strategy: {config_backtest['strategy']}")
print(f"Timeframe: {config_backtest['timeframe']}")


# Run the full backtest process
backtesting.start()

print("Backtest completed!")
print(f"Results available in backtesting.results")


Running backtest...
Strategy: WarriorMomentum
Timeframe: 5m


2025-07-21 14:55:10,459 - freqtrade.data.history.history_utils - INFO - Using indicator startup period: 200 ...

2025-07-21 14:55:10,489 - freqtrade.data.converter.converter - INFO - Missing data fillup for 1INCH/USD, 5m: before: 2700 - after: 8633 - 219.74%

2025-07-21 14:55:10,511 - freqtrade.data.converter.converter - INFO - Missing data fillup for AAVE/USD, 5m: before: 7967 - after: 8641 - 8.46%

2025-07-21 14:55:10,525 - freqtrade.data.history.datahandlers.idatahandler - INFO - Price jump in ACA/USD, 5m, spot between two candles of 11.79% detected.

...

                                              EXIT REASON STATS                                               
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Exit Reason ┃ Exits ┃ Avg Profit % ┃ Tot Profit USD ┃ Tot Profit % ┃ Avg Duration ┃  Win  Draw  Loss  Win% ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│       TOTAL │     0 │          0.0 │          0.000 │          0.0 │         0:00 │    0     0     0     0 │
└─────────────┴───────┴──────────────┴────────────────┴──────────────┴──────────────┴────────────────────────┘

                                                      MIXED TAG STATS                                                      
┏━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Enter Tag ┃ Exit Reason ┃ Trades ┃ Avg Profit % ┃ Tot Profit USD ┃ Tot Profit % ┃ Avg Duration ┃  Win  Draw  Loss  Win% ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│     TOTAL │             │      0 │          0.0 │          0.000 │          0.0 │         0:00 │    0     0     0     0 │
└───────────┴─────────────┴────────┴──────────────┴────────────────┴──────────────┴──────────────┴────────────────────────┘

No trades made. Your starting balance was 50000 USD, and your stake was unlimited.

Backtested 2025-06-19 09:45:00 -> 2025-07-19 00:25:00 | Max open trades : 10


                                                         STRATEGY SUMMARY                                                         
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃        Strategy ┃ Trades ┃ Avg Profit % ┃ Tot Profit USD ┃ Tot Profit % ┃ Avg Duration ┃  Win  Draw  Loss  Win% ┃     Drawdown ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ WarriorMomentum │      0 │         0.00 │          0.000 │          0.0 │         0:00 │    0     0     0     0 │ 0 USD  0.00% │
└─────────────────┴────────┴──────────────┴────────────────┴──────────────┴──────────────┴────────────────────────┴──────────────┘

Backtest completed!
Results available in backtesting.results


In [ ]:
# Open and display the backtest results file
import os
from pathlib import Path

# Get the backtest results directory
backtest_dir = Path(config_backtest["user_data_dir"]) / "backtest_results"

# Find the most recent backtest file
backtest_files = list(backtest_dir.glob("backtest-result-*.zip"))
if backtest_files:
    # Sort by modification time to get the most recent
    latest_backtest = max(backtest_files, key=os.path.getmtime)
    print(f"Opening latest backtest file: {latest_backtest}")

    # Load and display basic backtest statistics
    from freqtrade.data.btanalysis import load_backtest_stats

    stats = load_backtest_stats(latest_backtest)
    print("\n=== BACKTEST RESULTS SUMMARY ===")

    if 'strategy' in stats:
        strategy_stats = stats['strategy']
        for strategy_name, strategy_data in strategy_stats.items():
            print(f"\nStrategy: {strategy_name}")
            print(f"Total trades: {strategy_data.get('total_trades', 'N/A')}")
            print(f"Profit factor: {strategy_data.get('profit_factor', 'N/A')}")
            print(f"Total profit %: {strategy_data.get('profit_total_pct', 'N/A'):.2f}%")
            print(f"Max drawdown: {strategy_data.get('max_drawdown', 'N/A'):.2f}%")
            print(f"Win rate: {strategy_data.get('wins', 0) / max(strategy_data.get('total_trades', 1), 1) * 100:.1f}%")



else:
    print("No backtest result files found in:", backtest_dir)
    print("Available files:", list(backtest_dir.glob("*")))


Opening latest backtest file: /Users/conradlz/Documents/webclones/freqtrade/user_data/backtest_results/backtest-result-2025-07-21_14-56-27.zip


2025-07-21 14:58:16,423 - freqtrade.data.btanalysis.bt_fileutils - INFO - Loading backtest result from 
/Users/conradlz/Documents/webclones/freqtrade/user_data/backtest_results/backtest-result-2025-07-21_14-56-27.zip


=== BACKTEST RESULTS SUMMARY ===

Strategy: WarriorMomentum
Total trades: 0
Profit factor: 0.0
Error loading backtest stats: Unknown format code 'f' for object of type 'str'
Backtest file exists but may be empty or corrupted


In [82]:
# Load data using values set above
from freqtrade.data.history import load_pair_history

pair = "ALGO/USD"

candles = load_pair_history(
    datadir=data_location,
    timeframe=config["timeframe"],
    pair=pair,
    data_format="feather",  # Make sure to update this to your data
)

# Confirm success
print(f"Loaded {len(candles)} rows of data for {pair} from {data_location}")
candles.head()

2025-07-21 14:57:38,000 - freqtrade.data.converter.converter - INFO - Missing data fillup for ALGO/USD, 5m: before: 7484 - after: 8641 - 15.46%

Loaded 8641 rows of data for ALGO/USD from /Users/conradlz/Documents/webclones/freqtrade/user_data/data/kraken


,date,open,high,low,close,volume
0,2025-06-18 17:20:00+00:00,0.16714,0.16714,0.16710,0.16712,1417.998156
1,2025-06-18 17:25:00+00:00,0.16689,0.16689,0.16676,0.16676,6025.658647
2,2025-06-18 17:30:00+00:00,0.16621,0.16621,0.16584,0.16584,13110.534852
3,2025-06-18 17:35:00+00:00,0.16657,0.16722,0.16657,0.16722,11686.235005
4,2025-06-18 17:40:00+00:00,0.16746,0.16804,0.16746,0.16804,443.505957


In [83]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats
from freqtrade.plot.plotting import generate_candlestick_graph

backtest_dir = config["user_data_dir"] / "backtest_results"

# Limit graph period to keep plotly quick and reactive

# Load backtested trades as dataframe
trades = load_backtest_data(backtest_dir)

# Show value-counts per pair
trades.groupby("pair")["exit_reason"].value_counts()

# Filter trades to one pair
trades_red = trades.loc[trades["pair"] == pair]

# Use candles data instead of undefined 'data' variable
data_red = candles
# Generate candlestick graph
graph = generate_candlestick_graph(
    pair=pair,
    data=data_red,
    trades=trades_red,
    indicators1=["ema9", "ema21", "ema50", "bb_upperband", "bb_lowerband", "bb_middleband"],
    indicators2=["rsi", "macd", "macdsignal", "macdhist", "adx", "mfi"],
    plot_config={'show_date': True}
)

# Show graph inline
graph.show()

# Render graph in a separate window
# graph.show(renderer="browser")

2025-07-21 14:57:38,911 - freqtrade.data.btanalysis.bt_fileutils - INFO - Loading backtest result from 
/Users/conradlz/Documents/webclones/freqtrade/user_data/backtest_results/backtest-result-2025-07-21_14-56-27.zip

KeyError: 'pair'